# French Property Intelligence
## 02 — Modeling Dataset

### Objective

This notebook transforms the audited transaction data into a reproducible modeling dataset for residential sale-price prediction in metropolitan France.

The modeling dataset is designed once and then reused across multiple experiments so that baseline and advanced models can be compared on the same observations and temporal validation framework.

The notebook focuses on:

1. reproducing the confirmed data-cleaning rules;
2. defining the prediction target;
3. selecting features that are available and appropriate at prediction time;
4. preventing target leakage and identifier memorization;
5. engineering a compact set of property, temporal, and geographic features;
6. defining fixed temporal train, validation, and test populations;
7. preparing reusable datasets for subsequent model experiments.

Model fitting and model comparison are intentionally performed in later notebooks.

In [27]:
# PURPOSE:
# Import the libraries required for modeling-dataset construction
# and define the project data paths.

from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd().parent
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

TRANSACTIONS_PATH = DATA_RAW_DIR / "transactions.npz"

DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data:     {TRANSACTIONS_PATH}")
print(f"Processed:    {DATA_PROCESSED_DIR}")

Project root: c:\Users\Alex\Desktop\Jehda AI\Fullstack\CDSD Certification\6_french-property-intelligence
Raw data:     c:\Users\Alex\Desktop\Jehda AI\Fullstack\CDSD Certification\6_french-property-intelligence\data\raw\transactions.npz
Processed:    c:\Users\Alex\Desktop\Jehda AI\Fullstack\CDSD Certification\6_french-property-intelligence\data\processed


In [28]:
# PURPOSE:
# Create a PyArrow Parquet reader for the transaction dataset.
#
# Despite its .npz extension, the data audit established that
# transactions.npz is actually an Apache Parquet file.

parquet_file = pq.ParquetFile(TRANSACTIONS_PATH)

print(f"Raw rows:    {parquet_file.metadata.num_rows:,}")
print(f"Raw columns: {parquet_file.metadata.num_columns}")
print(f"Row groups:  {parquet_file.num_row_groups}")

Raw rows:    9,141,573
Raw columns: 20
Row groups:  9


### Modeling principles

The dataset construction follows several principles established during the data audit:

- **Target:** total transaction price (`prix`).
- **Scope:** residential houses and apartments in metropolitan France.
- **Validation:** chronological rather than purely random, because the production task requires generalization to later transactions.
- **Leakage prevention:** features derived from transaction prices must never use information from the observation being predicted or from future transactions.
- **Identifiers:** transaction IDs, exact addresses, and cadastral parcel IDs are not used as direct predictors.
- **Conservative cleaning:** unusual observations are not removed solely because they lie in an extreme national percentile.
- **Comparability:** all candidate models should be evaluated on the same fixed validation and test populations.
- **Production compatibility:** predictive features and transformations should be reproducible from information available when the API receives a valuation request. Notebook-only transformations that cannot be reproduced in production are avoided.
- **Time as context, not forecasting:** the project is formulated as supervised tabular regression rather than time-series forecasting. Transaction/valuation time may be used as market context, while chronological validation is used to measure generalization to later transactions.

## 1. Feature eligibility and production design

Before engineering features, each raw variable is evaluated according to five questions:

1. Does it contain potentially useful predictive information?
2. Would the information be available when valuing a new property?
3. Could it introduce target leakage or memorization?
4. Is it compatible with the intended user experience and privacy constraints?
5. Can the same transformation be reproduced reliably in the production API?

The objective is not to maximize the number of variables, but to build a compact and defensible feature set that can be reproduced at inference time.

### Raw-variable eligibility matrix

| Raw variable | Decision | Reason |
|---|---|---|
| `id_transaction` | **Exclude** | Database identifier; unavailable for a new valuation and may encourage memorization |
| `date_transaction` | **Transform / evaluate** | Filtering only / exclude from initial predictors** | Used to restrict the dataset to a recent market window. The initial product is a cross-sectional valuation model rather than a time-series forecasting model, so transaction date is not provided to the model. |
| `prix` | **Target** | Sale price to predict |
| `id_ville` | **Test later** | Fine geographic information but high cardinality and database-specific identifier |
| `code_postal` | **Candidate** | Useful geographic information; can potentially be derived from the address |
| `vefa` | **Candidate** | Property characteristic known at prediction time |
| `n_pieces` | **Candidate** | Important property characteristic; zero values treated as unknown |
| `surface_habitable` | **Candidate** | Core property characteristic |
| `latitude` | **Candidate** | Fine-grained location; derived automatically from user address |
| `longitude` | **Candidate** | Fine-grained location; derived automatically from user address |
| `departement` | **Candidate / context** | Useful regional information but too coarse to represent location alone |
| `ville` | **Test later** | Potentially useful location information but high-cardinality |
| `adresse` | **Transform, not direct predictor** | User-friendly location input; should be geocoded rather than learned as free text |
| `type_batiment` | **Candidate** | Core distinction between house and apartment |
| `id_parcelle_cadastre` | **Exclude initially** | Very high-cardinality identifier with memorization/privacy/generalization concerns |
| `surface_dependances` | **Transform / test** | Potential property information but encoded as structured strings and often empty |
| `surface_locaux_industriels` | **Transform / test** | Sparse and potentially less relevant to residential valuation |
| `surface_terrains_agricoles` | **Transform / test** | Potentially useful for some houses but very sparse |
| `surface_terrains_sols` | **Transform / test** | Potential land information; requires decoding |
| `surface_terrains_nature` | **Transform / test** | Potential land information; requires decoding and is highly sparse |

### Initial modeling strategy

The first modeling dataset will prioritize reliable features that are both predictive and reproducible in production:

- habitable surface;
- number of rooms;
- property type;
- VEFA status;
- fine geographic coordinates derived from the property address;
- selected administrative geographic information;


The encoded additional-surface variables will not be discarded automatically. Their structure will first be decoded and their incremental predictive value can then be tested through controlled model comparisons.

Exact address, cadastral parcel ID, and transaction ID will not be direct model predictors.

The intended product is a **tabular property valuation system**, not a time-series forecasting system. The end user provides property characteristics and an address; geographic model features are derived automatically by the application.

## 2. Additional property-surface variables

Several raw variables contain potentially useful information about dependencies and land, but they are stored as encoded strings such as `{}`, `{0}`, or `{2387}` rather than directly as numeric values.

Before including these variables in the modeling dataset, their structure is inspected to determine whether they can be transformed into meaningful and production-compatible numeric features.

This is a targeted data-preparation check rather than additional exploratory analysis.

In [29]:
# PURPOSE:
# Inspect the encoded surface-related fields before deciding whether
# they can be transformed into useful numeric model features.
#
# Only one Parquet row group and five columns are loaded because
# we only need to understand their encoding at this stage.

surface_columns = [
    "surface_dependances",
    "surface_locaux_industriels",
    "surface_terrains_agricoles",
    "surface_terrains_sols",
    "surface_terrains_nature",
]

surface_sample = parquet_file.read_row_group(
    0,
    columns=surface_columns
).to_pandas()


for col in surface_columns:
    print(f"\n{'=' * 60}")
    print(col)
    print(f"{'=' * 60}")

    print(surface_sample[col].value_counts().head(15))


surface_dependances
surface_dependances
{}                                 641679
{0}                                279882
{0,0}                               98685
{0,0,0}                             22962
{0,0,0,0}                            3931
{0,0,0,0,0}                           852
{0,0,0,0,0,0}                         257
{0,0,0,0,0,0,0}                        89
{0,0,0,0,0,0,0,0}                      40
{0,0,0,0,0,0,0,0,0}                    27
{0,0,0,0,0,0,0,0,0,0}                  21
{0,0,0,0,0,0,0,0,0,0,0}                16
{NULL,0}                                9
{0,0,0,0,0,0,0,0,0,0,0,0}               8
{0,0,0,0,0,0,0,0,0,0,0,0,0,0,0}         8
Name: count, dtype: int64

surface_locaux_industriels
surface_locaux_industriels
{}        1030358
{0}          1675
{50}          370
{30}          357
{60}          309
{40}          304
{20}          287
{100}         287
{NULL}        274
{70}          244
{80}          241
{35}          192
{45}          190
{120}         

### Decision

The additional surface variables are stored as collections rather than simple numeric measurements. Empty collections (`{}`), zero-valued collections (`{0}`, `{0,0}`, etc.), and non-zero values are observed.

Their exact interpretation and appropriate aggregation cannot be assumed from the encoding alone. In addition, several of these fields are highly sparse.

Therefore:

- the five encoded `surface_*` variables are **excluded from the initial baseline feature set**;
- the raw information is preserved;
- they may later be transformed and evaluated as an enrichment experiment;
- they will only be retained if a controlled model comparison demonstrates incremental predictive value.

This keeps the first production pipeline compact and avoids introducing transformations whose semantics are not yet sufficiently established.

## 3. Definition of the recent market window

The valuation product is designed to estimate property values from characteristics and location rather than forecast a future price series.

Using the complete 2014–2024 history could mix substantially different market regimes. Therefore, the modeling population is restricted to a more recent period.

The transaction date is used here to define the modeling population, but it is not included as a predictor in the initial model.

Several recent windows are compared before selecting the final modeling period.

In [30]:
# PURPOSE:
# Compare several recent historical windows before fixing the modeling
# population used by all subsequent models.
#
# Only transaction dates are loaded, keeping this decision inexpensive.

date_df = parquet_file.read(
    columns=["date_transaction"]
).to_pandas()

date_df["year"] = date_df["date_transaction"].dt.year

window_summary = []

for start_year in [2019, 2020, 2021]:
    mask = date_df["year"].between(start_year, 2024)

    window_summary.append({
        "window": f"{start_year}–2024 H1",
        "start_date": date_df.loc[mask, "date_transaction"].min(),
        "end_date": date_df.loc[mask, "date_transaction"].max(),
        "rows": int(mask.sum()),
        "share_of_raw_data": mask.mean()
    })

window_summary = pd.DataFrame(window_summary)

window_summary["share_of_raw_data"] = (
    window_summary["share_of_raw_data"] * 100
).round(2)

window_summary

,window,start_date,end_date,rows,share_of_raw_data
0,2019–2024 H1,2019-01-01,2024-06-30,5176582,56.63
1,2020–2024 H1,2020-01-01,2024-06-30,4197082,45.91
2,2021–2024 H1,2021-01-01,2024-06-30,3252457,35.58


### Selected modeling window

The initial modeling population is restricted to transactions from **1 January 2020 through 30 June 2024**.

This window contains approximately **4.20 million raw transactions**, representing 45.9% of the complete transaction dataset. It provides a compromise between:

- focusing the model on relatively recent market conditions;
- retaining several million observations for geographic and property diversity;
- avoiding unnecessary dependence on transactions from substantially older market periods.

The cutoff is a modeling design choice rather than an assumption that market conditions were perfectly stable throughout 2020–2024.

The initial model is formulated as a **cross-sectional property valuation model**. `date_transaction` is therefore used to select the modeling population but is not included as an initial predictor.

Train, validation, and test observations will subsequently be randomly sampled from this same recent-market population using a fixed random seed, ensuring that all candidate models are compared on identical data partitions.

## 4. Domain-plausibility checks before final cleaning

The initial audit identified a very small number of physically or economically implausible observations.

Before fixing the modeling population, candidate cleaning rules are evaluated on the selected 2020–2024 metropolitan market window.

The objective is not statistical outlier trimming. Observations are only considered for exclusion when there is a defensible property-market or physical-consistency rationale.

In particular, extreme sale prices are not removed simply because they belong to the upper tail: genuinely expensive residential properties exist. Target-derived measures such as price per square metre are therefore used primarily as diagnostics rather than automatic filtering rules.

In [31]:
# PURPOSE:
# Load only the variables needed to evaluate domain-plausibility cleaning
# rules on the selected recent metropolitan modeling window.
#
# No final cleaning thresholds are applied in this cell.

diagnostic_columns = [
    "date_transaction",
    "prix",
    "surface_habitable",
    "n_pieces",
    "type_batiment",
    "departement",
]

diagnostic_df = parquet_file.read(
    columns=diagnostic_columns
).to_pandas()


# Restrict to the selected recent market period.
diagnostic_df = diagnostic_df.loc[
    diagnostic_df["date_transaction"].between(
        "2020-01-01",
        "2024-06-30"
    )
].copy()


# Restrict to metropolitan France according to the scope decision
# established in notebook 01.
overseas_departments = {"971", "972", "973", "974"}

diagnostic_df = diagnostic_df.loc[
    ~diagnostic_df["departement"].isin(overseas_departments)
].copy()


print(f"Recent metropolitan transactions: {len(diagnostic_df):,}")

Recent metropolitan transactions: 4,151,839


In [32]:
# PURPOSE:
# Quantify candidate market/physical plausibility problems before deciding
# which rules should become part of the modeling-data cleaning pipeline.
#
# These are diagnostics only: this cell does not remove any observations.

valid_surface = diagnostic_df["surface_habitable"] > 0
valid_rooms = diagnostic_df["n_pieces"] > 0
valid_price = diagnostic_df["prix"] > 0

diagnostic_df["price_per_m2"] = np.where(
    valid_surface,
    diagnostic_df["prix"] / diagnostic_df["surface_habitable"],
    np.nan
)

diagnostic_df["surface_per_room"] = np.where(
    valid_surface & valid_rooms,
    diagnostic_df["surface_habitable"] / diagnostic_df["n_pieces"],
    np.nan
)


checks = {
    # Target
    "price <= 0": diagnostic_df["prix"] <= 0,
    "0 < price < €10k": (
        (diagnostic_df["prix"] > 0) &
        (diagnostic_df["prix"] < 10_000)
    ),

    # Habitable surface
    "surface <= 0 m²": diagnostic_df["surface_habitable"] <= 0,
    "0 < surface < 10 m²": (
        (diagnostic_df["surface_habitable"] > 0) &
        (diagnostic_df["surface_habitable"] < 10)
    ),
    "10 <= surface < 15 m²": (
        (diagnostic_df["surface_habitable"] >= 10) &
        (diagnostic_df["surface_habitable"] < 15)
    ),
    "surface > 500 m²": diagnostic_df["surface_habitable"] > 500,

    # Rooms
    "rooms == 0": diagnostic_df["n_pieces"] == 0,
    "rooms > 15": diagnostic_df["n_pieces"] > 15,
    "rooms > 20": diagnostic_df["n_pieces"] > 20,

    # Cross-variable diagnostics
    "surface/room < 5 m²": (
        diagnostic_df["surface_per_room"] < 5
    ),
    "price/m² < €100": (
        (diagnostic_df["price_per_m2"] > 0) &
        (diagnostic_df["price_per_m2"] < 100)
    ),
    "price/m² > €50k": (
        diagnostic_df["price_per_m2"] > 50_000
    ),
}


plausibility_summary = pd.DataFrame([
    {
        "diagnostic": name,
        "rows": int(mask.sum()),
        "share_pct": round(mask.mean() * 100, 4),
    }
    for name, mask in checks.items()
]).sort_values("rows", ascending=False)

plausibility_summary

,diagnostic,rows,share_pct
1,0 < price < €10k,20973,0.5051
10,price/m² < €100,14977,0.3607
4,10 <= surface < 15 m²,13185,0.3176
6,rooms == 0,3975,0.0957
3,0 < surface < 10 m²,2785,0.0671
11,price/m² > €50k,1537,0.0370
9,surface/room < 5 m²,702,0.0169
5,surface > 500 m²,580,0.0140
7,rooms > 15,478,0.0115
8,rooms > 20,252,0.0061


### Cleaning policy for the initial modeling population

Based on the data audit and the recent-market plausibility diagnostics, the following rules are applied:

- retain transactions with strictly positive sale prices;
- retain residential properties with habitable surface between **10 m² and 500 m²**, inclusive;
- interpret `n_pieces = 0` as unavailable rather than as a literal zero-room dwelling;
- interpret `n_pieces = 0` or `n_pieces > 15` as unavailable/unreliable and replace it with a missing value rather than discarding the transaction;
- do not remove observations solely because their sale price or price per square metre is statistically extreme.

The surface bounds define the scope of the initial automated residential valuation product and remove a very small number of physically atypical or previously identified inconsistent records.

Target-derived indicators such as price per square metre are retained as **data-quality diagnostics only** and are not used as automatic filtering criteria. This avoids improving model metrics artificially through aggressive target-based trimming.

The effect of remaining unusual transactions will be assessed later through validation metrics and model error analysis.

In [33]:
# PURPOSE:
# Inspect the very small set of transactions with more than 20 rooms
# before deciding whether the room count should be treated as unreliable.
#
# This is a targeted validation of a proposed cleaning rule.

extreme_rooms = (
    diagnostic_df.loc[
        diagnostic_df["n_pieces"] > 20,
        [
            "prix",
            "surface_habitable",
            "n_pieces",
            "type_batiment",
            "departement",
            "surface_per_room",
            "price_per_m2",
        ]
    ]
    .sort_values("n_pieces", ascending=False)
)

extreme_rooms.head(20)

,prix,surface_habitable,n_pieces,type_batiment,departement,surface_per_room,price_per_m2
8251988,119560.0,61,109,Maison,91,0.559633,1960.000000
7456762,233000.0,92,95,Maison,81,0.968421,2532.608696
2098958,153000.0,87,90,Appartement,29,0.966667,1758.620690
6831965,211150.0,89,89,Maison,76,1.000000,2372.471910
8105691,152000.0,82,84,Maison,89,0.976190,1853.658537
8288495,775000.0,171,80,Maison,91,2.137500,4532.163743
8860227,419000.0,98,78,Appartement,94,1.256410,4275.510204
5347065,91500.0,68,70,Appartement,62,0.971429,1345.588235
7020267,360500.0,115,70,Maison,77,1.642857,3134.782609
2183189,320000.0,101,70,Maison,30,1.442857,3168.316832


Cleaning Policy v1:

Recent period        2020-01-01 → 2024-06-30

Geography            Metropolitan France

Price                > €0

Habitable surface    10–500 m² inclusive

Rooms                 0 or >15 → missing

Extreme price         retained

Extreme €/m²          retained

Address               not direct predictor

Transaction ID        not predictor

Cadastral ID          not predictor

Encoded surfaces      baseline excluded / test later

Date                  filtering only / baseline excluded

## 5. Construction of the baseline modeling dataset

A single cleaned modeling population is first constructed using the rules established above.

The population is subsequently separated into **houses (`Maison`)** and **apartments (`Appartement`)**. These property types will initially be modeled separately because their valuation mechanisms may differ substantially.

Using a common cleaning pipeline before separating the populations ensures that both datasets follow identical scope and data-quality rules.

The value of specialized models will later be verified quantitatively against model performance rather than assumed from domain intuition alone.

## Cleaning Policy v2 — Residential transaction consistency

The initial modeling experiments revealed that a very small number of transactions have prices that are extremely inconsistent with the characteristics of the individual dwelling represented in the dataset.

Examples include residential units with ordinary habitable surfaces associated with transaction values of tens or hundreds of millions of euros.

The objective of the product is not to predict every transaction recorded in the source data. It is to provide an indicative and reasonable property valuation for an individual residential house or apartment in metropolitan France, primarily as decision support for private buyers and sellers.

Therefore, before constructing the definitive modeling population, additional transaction-consistency rules are investigated.

These rules must:

- reflect the intended residential valuation use case;
- remove clearly implausible property/transaction combinations rather than optimize validation scores;
- preserve legitimate geographic and market variation;
- avoid aggressive percentile-based target trimming;
- be defined before regenerating the train, validation and test populations.

The following analysis investigates price-per-square-metre distributions separately for houses and apartments before Cleaning Policy v2 is finalized.

In [34]:
# PURPOSE:
# Examine price-per-m² consistency BEFORE finalizing Cleaning Policy v2.
#
# This diagnostic reconstructs the same candidate population used by the
# master modeling dataset:
#   - transactions from 2020-01-01 to 2024-06-30
#   - metropolitan France
#   - positive sale price
#   - habitable surface between 10 and 500 m²
#
# IMPORTANT:
# This cell is diagnostic only.
# It does NOT modify model_df and does NOT remove any observations.

diagnostic_columns = [
    "date_transaction",
    "prix",
    "surface_habitable",
    "type_batiment",
    "departement",
]

diagnostic_df = parquet_file.read(
    columns=diagnostic_columns
).to_pandas()


# Apply the SAME existing scope rules used by the modeling dataset.
diagnostic_df = diagnostic_df.loc[
    diagnostic_df["date_transaction"].between(
        "2020-01-01",
        "2024-06-30"
    )
].copy()

overseas_departments = {"971", "972", "973", "974"}

diagnostic_df = diagnostic_df.loc[
    ~diagnostic_df["departement"].isin(overseas_departments)
].copy()

diagnostic_df = diagnostic_df.loc[
    diagnostic_df["prix"] > 0
].copy()

diagnostic_df = diagnostic_df.loc[
    diagnostic_df["surface_habitable"].between(10, 500)
].copy()


# Price per m² is used here ONLY as a diagnostic consistency measure.
diagnostic_df["price_per_m2"] = (
    diagnostic_df["prix"]
    / diagnostic_df["surface_habitable"]
)


price_m2_quantiles = [
    0.0001,
    0.001,
    0.005,
    0.01,
    0.50,
    0.99,
    0.995,
    0.999,
    0.9999,
    1.0,
]


for property_type in ["Maison", "Appartement"]:

    values = diagnostic_df.loc[
        diagnostic_df["type_batiment"] == property_type,
        "price_per_m2",
    ]

    print(f"\n{'=' * 60}")
    print(property_type.upper())
    print(f"{'=' * 60}")

    print(f"Transactions: {len(values):,}")

    print("\nPrice/m² quantiles:")
    print(values.quantile(price_m2_quantiles))

    print("\nLower-tail counts:")
    print(f"< €50/m²:    {(values < 50).sum():,}")
    print(f"< €100/m²:   {(values < 100).sum():,}")
    print(f"< €250/m²:   {(values < 250).sum():,}")
    print(f"< €500/m²:   {(values < 500).sum():,}")

    print("\nUpper-tail counts:")
    print(f"> €20k/m²:   {(values > 20_000).sum():,}")
    print(f"> €30k/m²:   {(values > 30_000).sum():,}")
    print(f"> €50k/m²:   {(values > 50_000).sum():,}")
    print(f"> €100k/m²:  {(values > 100_000).sum():,}")


MAISON
Transactions: 2,339,916

Price/m² quantiles:
0.0001    8.000000e-03
0.0010    7.246377e+00
0.0050    1.233582e+02
0.0100    2.083333e+02
0.5000    2.054544e+03
0.9900    9.677419e+03
0.9950    1.232475e+04
0.9990    2.410804e+04
0.9999    9.410739e+04
1.0000    6.072185e+06
Name: price_per_m2, dtype: float64

Lower-tail counts:
< €50/m²:    5,369
< €100/m²:   9,256
< €250/m²:   29,915
< €500/m²:   95,787

Upper-tail counts:
> €20k/m²:   3,471
> €30k/m²:   1,539
> €50k/m²:   640
> €100k/m²:  211

APPARTEMENT
Transactions: 1,808,557

Price/m² quantiles:
0.0001    1.010101e-02
0.0010    1.162791e+01
0.0050    2.235175e+02
0.0100    4.710145e+02
0.5000    3.266667e+03
0.9900    1.452941e+04
0.9950    1.690969e+04
0.9990    2.821399e+04
0.9999    1.333333e+05
1.0000    1.593750e+07
Name: price_per_m2, dtype: float64

Lower-tail counts:
< €50/m²:    3,843
< €100/m²:   5,662
< €250/m²:   9,588
< €500/m²:   19,471

Upper-tail counts:
> €20k/m²:   4,835
> €30k/m²:   1,559
> €50k/m²:   6

In [35]:
# PURPOSE:
# Inspect concrete transactions from the extreme price-per-m² tails
# before defining Cleaning Policy v2.
#
# We do not want to choose cleaning thresholds only from quantiles.
# Looking at actual rows helps distinguish legitimate unusual properties
# from transactions that are inconsistent with an individual residential
# valuation.
#
# DIAGNOSTIC ONLY:
# Nothing is removed from the modeling dataset here.

inspection_columns = [
    "date_transaction",
    "prix",
    "surface_habitable",
    "n_pieces",
    "type_batiment",
    "vefa",
    "code_postal",
    "departement",
    "latitude",
    "longitude",
]

inspection_df = parquet_file.read(
    columns=inspection_columns
).to_pandas()

# Same candidate population as the future modeling dataset
inspection_df = inspection_df.loc[
    inspection_df["date_transaction"].between(
        "2020-01-01",
        "2024-06-30"
    )
].copy()

inspection_df = inspection_df.loc[
    ~inspection_df["departement"].isin(
        {"971", "972", "973", "974"}
    )
].copy()

inspection_df = inspection_df.loc[
    (inspection_df["prix"] > 0)
    & inspection_df["surface_habitable"].between(10, 500)
].copy()

inspection_df["price_per_m2"] = (
    inspection_df["prix"]
    / inspection_df["surface_habitable"]
)


for property_type in ["Maison", "Appartement"]:

    subset = inspection_df.loc[
        inspection_df["type_batiment"] == property_type
    ]

    print(f"\n{'=' * 70}")
    print(f"{property_type.upper()} — LOWEST PRICE/m²")
    print(f"{'=' * 70}")

    display(
        subset.nsmallest(15, "price_per_m2")[
            inspection_columns + ["price_per_m2"]
        ]
    )

    print(f"\n{'=' * 70}")
    print(f"{property_type.upper()} — HIGHEST PRICE/m²")
    print(f"{'=' * 70}")

    display(
        subset.nlargest(15, "price_per_m2")[
            inspection_columns + ["price_per_m2"]
        ]
    )


MAISON — LOWEST PRICE/m²


,date_transaction,prix,surface_habitable,n_pieces,type_batiment,vefa,code_postal,departement,latitude,longitude,price_per_m2
3889882,2022-01-31,1.0,390,9,Maison,False,44510,44,47.267714,-2.423188,0.002564
444581,2021-12-31,1.0,362,8,Maison,False,6100,06,43.737146,7.241763,0.002762
215545,2022-05-16,1.0,360,10,Maison,False,4110,04,43.854874,5.646799,0.002778
687501,2020-10-14,1.0,350,6,Maison,False,11120,11,43.240918,2.925598,0.002857
7663753,2022-02-04,1.0,335,7,Maison,False,83380,83,43.343213,6.695034,0.002985
4099031,2022-11-18,1.2,400,9,Maison,False,47400,47,44.391094,0.303051,0.003000
208843,2020-06-26,1.0,330,7,Maison,False,4100,04,43.841264,5.791636,0.003030
6121925,2022-02-07,1.0,311,15,Maison,False,72610,72,48.347449,0.130792,0.003215
2868948,2020-02-06,1.0,305,10,Maison,False,34270,34,43.834737,3.884511,0.003279
2930370,2022-07-20,1.0,305,10,Maison,False,34270,34,43.834737,3.884511,0.003279



MAISON — HIGHEST PRICE/m²


,date_transaction,prix,surface_habitable,n_pieces,type_batiment,vefa,code_postal,departement,latitude,longitude,price_per_m2
4993783,2022-10-28,722590020.0,119,5,Maison,False,59700,59,50.662116,3.097775,6.072185e+06
2229074,2023-11-29,99706800.0,20,1,Maison,False,30300,30,43.789482,4.632335,4.985340e+06
7255726,2022-09-08,467200000.0,122,7,Maison,False,78700,78,48.995242,2.114489,3.829508e+06
8248456,2020-10-14,435244992.0,122,5,Maison,False,91800,91,48.707877,2.515844,3.567582e+06
2191744,2020-11-25,60000000.0,20,1,Maison,True,30300,30,43.789482,4.632335,3.000000e+06
1758416,2022-06-27,240000000.0,94,5,Maison,False,25170,25,47.272483,5.915844,2.553191e+06
5312634,2022-03-09,189800000.0,86,4,Maison,False,62990,62,50.455947,1.943743,2.206977e+06
251518,2023-09-19,56071200.0,45,3,Maison,True,5240,05,44.952464,6.551612,1.246027e+06
8465364,2020-12-21,94800000.0,90,4,Maison,True,92220,92,48.805938,2.325201,1.053333e+06
6205630,2021-10-29,60000000.0,60,4,Maison,False,73120,73,45.412277,6.636720,1.000000e+06



APPARTEMENT — LOWEST PRICE/m²


,date_transaction,prix,surface_habitable,n_pieces,type_batiment,vefa,code_postal,departement,latitude,longitude,price_per_m2
7761783,2020-03-23,0.15,72,3,Appartement,False,84400,84,43.883547,5.363615,0.002083
6669486,2022-12-14,1.00,460,5,Appartement,False,75011,75,48.851418,2.393182,0.002174
7246047,2022-04-08,0.15,43,2,Appartement,False,78330,78,48.809313,2.050083,0.003488
8702230,2023-09-20,1.00,277,6,Appartement,False,93160,93,48.840101,2.543578,0.003610
6575078,2020-01-06,0.15,40,2,Appartement,False,75018,75,48.885802,2.338371,0.003750
4926772,2021-02-18,1.00,241,6,Appartement,False,59800,59,50.643216,3.054994,0.004149
713208,2023-09-12,0.15,36,1,Appartement,False,11000,11,43.206312,2.386460,0.004167
6105136,2020-05-22,1.00,230,5,Appartement,False,72000,72,48.005515,0.193687,0.004348
242812,2021-05-04,1.00,226,3,Appartement,False,5290,05,44.817256,6.482759,0.004425
3349932,2020-01-13,1.00,220,5,Appartement,False,38190,38,45.258732,5.914782,0.004545



APPARTEMENT — HIGHEST PRICE/m²


,date_transaction,prix,surface_habitable,n_pieces,type_batiment,vefa,code_postal,departement,latitude,longitude,price_per_m2
6712593,2024-06-27,255000000.0,16,2,Appartement,False,75008,75,48.867252,2.307422,1.593750e+07
6664860,2022-10-20,63695420.0,14,1,Appartement,True,75011,75,48.858919,2.387437,4.549673e+06
6676709,2023-03-09,157200000.0,36,2,Appartement,False,75008,75,48.871820,2.298186,4.366667e+06
6695183,2023-10-20,135000000.0,31,1,Appartement,False,75008,75,48.868161,2.324552,4.354839e+06
6662073,2022-09-26,131027072.0,34,2,Appartement,False,75017,75,48.880985,2.311020,3.853737e+06
8247547,2020-09-30,177600000.0,62,3,Appartement,False,91000,91,48.618261,2.430744,2.864516e+06
6679379,2023-04-12,134400000.0,51,2,Appartement,True,75008,75,48.876994,2.303613,2.635294e+06
6659501,2022-09-05,55200000.0,28,1,Appartement,True,75018,75,48.891364,2.331246,1.971429e+06
6639023,2022-02-09,36350000.0,19,1,Appartement,False,75009,75,48.874373,2.335509,1.913158e+06
6695412,2023-10-25,53500000.0,33,3,Appartement,False,75008,75,48.873428,2.315489,1.621212e+06


In [36]:
# PURPOSE:
# Construct the master modeling dataset from production-compatible
# property and geographic variables.
#
# The same cleaning rules are applied before separating houses and
# apartments, ensuring consistent populations across later experiments.
#
# Cleaning Policy v2 additionally removes transactions whose recorded
# price is strongly inconsistent with the habitable surface represented
# by the row. This aligns the modeling population with the product goal:
# estimating a reasonable sale value for an individual residential
# property for buyer/seller decision support.
#
# IMPORTANT:
# price_per_m2 is used ONLY as a data-quality / consistency criterion.
# Because it is calculated from the target price, it MUST NOT become
# a predictive feature.

modeling_columns = [
    "date_transaction",
    "prix",
    "surface_habitable",
    "n_pieces",
    "type_batiment",
    "vefa",
    "latitude",
    "longitude",
    "code_postal",
    "departement",
]

model_df = parquet_file.read(
    columns=modeling_columns
).to_pandas()


# 1. Recent market window
# Keep transactions representative of the recent market period selected
# for this project: January 2020 through June 2024.
model_df = model_df.loc[
    model_df["date_transaction"].between(
        "2020-01-01",
        "2024-06-30"
    )
].copy()


# 2. Metropolitan product scope
# The product is currently scoped to metropolitan France.
overseas_departments = {"971", "972", "973", "974"}

model_df = model_df.loc[
    ~model_df["departement"].isin(overseas_departments)
].copy()


# 3. Valid positive target
# Zero or negative transaction prices cannot represent the residential
# sale value that the model is intended to estimate.
model_df = model_df.loc[
    model_df["prix"] > 0
].copy()


# 4. Product-scope / plausible habitable surface
# Keep residential properties between 10 and 500 m².
# This removes physically implausible or clearly out-of-scope surfaces
# while retaining a broad range of residential properties.
model_df = model_df.loc[
    model_df["surface_habitable"].between(
        10,
        500,
        inclusive="both"
    )
].copy()


# 5. Residential transaction-consistency rule
#
# Diagnostic analysis showed a small number of transactions with prices
# that are extremely inconsistent with the habitable surface represented
# by the row (for example, ordinary residential surfaces associated with
# nominal prices or transaction values of tens/hundreds of millions).
#
# We retain a deliberately broad range of:
#     €100/m² <= transaction price per m² <= €30,000/m²
#
# The objective is NOT to remove expensive properties simply because
# they are outliers. The broad upper limit preserves legitimate luxury
# residential transactions while excluding extreme price/surface
# combinations that are inconsistent with the individual-property
# valuation use case.

model_df["price_per_m2"] = (
    model_df["prix"]
    / model_df["surface_habitable"]
)

before_consistency_filter = len(model_df)

model_df = model_df.loc[
    model_df["price_per_m2"].between(
        100,
        30_000,
        inclusive="both"
    )
].copy()

removed_consistency = (
    before_consistency_filter - len(model_df)
)

removed_consistency_pct = (
    removed_consistency
    / before_consistency_filter
    * 100
)

print(
    "Removed by residential transaction-consistency rule:",
    f"{removed_consistency:,}",
    f"({removed_consistency_pct:.3f}%)"
)


# price_per_m2 contains information derived directly from the target.
# It was created only for cleaning and must NOT remain in the modeling
# dataset because that would create target leakage.
model_df = model_df.drop(
    columns=["price_per_m2"]
)


# 6. Treat unreliable room counts as missing rather than discarding
# otherwise useful transactions.
#
# A room count of 0 or greater than 15 was previously identified as
# unreliable in this dataset. We preserve the transaction but represent
# the room count as missing so that model preprocessing can handle it.
model_df["n_pieces"] = model_df["n_pieces"].astype(float)

model_df.loc[
    (model_df["n_pieces"] == 0)
    | (model_df["n_pieces"] > 15),
    "n_pieces"
] = np.nan


# Final population checks
print(f"\nClean master population: {len(model_df):,}")

print("\nProperty types:")
print(model_df["type_batiment"].value_counts())

print("\nDate range:")
print(
    model_df["date_transaction"].min(),
    "→",
    model_df["date_transaction"].max()
)

Removed by residential transaction-consistency rule: 18,016 (0.434%)

Clean master population: 4,130,457

Property types:
type_batiment
Maison         2329121
Appartement    1801336
Name: count, dtype: int64

Date range:
2020-01-01 00:00:00 → 2024-06-30 00:00:00


## 6. Baseline feature schema

The cleaned master population is separated into house and apartment populations.

Because the property type is constant within each specialized dataset, `type_batiment` is used to route a prediction to the appropriate model but is not itself used as a predictor inside the house or apartment model.

Before creating the fixed train/validation/test partitions, the candidate predictor schema is validated for missing values, data types, and categorical cardinality.

In [37]:
# PURPOSE:
# Derive the two specialized modeling populations from the same cleaned
# master dataset.
#
# We keep type_batiment temporarily for traceability, although it will not
# be a predictor because each specialized model contains only one type.

houses_df = (
    model_df.loc[model_df["type_batiment"] == "Maison"]
    .copy()
    .reset_index(drop=True)
)

apartments_df = (
    model_df.loc[model_df["type_batiment"] == "Appartement"]
    .copy()
    .reset_index(drop=True)
)

print(f"Houses:     {len(houses_df):,}")
print(f"Apartments: {len(apartments_df):,}")
print(f"Combined:   {len(houses_df) + len(apartments_df):,}")

Houses:     2,329,121
Apartments: 1,801,336
Combined:   4,130,457


In [38]:
# PURPOSE:
# Validate missingness, data types, and cardinality of the candidate
# baseline predictors before fixing the model feature schema.
#
# This helps identify variables that require preprocessing and prevents
# accidental treatment of identifiers such as postal codes as continuous
# numeric quantities.

candidate_features = [
    "surface_habitable",
    "n_pieces",
    "vefa",
    "latitude",
    "longitude",
    "code_postal",
    "departement",
]

feature_schema = pd.DataFrame({
    "dtype": model_df[candidate_features].dtypes.astype(str),
    "missing_rows": model_df[candidate_features].isna().sum(),
    "missing_pct": (
        model_df[candidate_features].isna().mean() * 100
    ).round(4),
    "unique_values": model_df[candidate_features].nunique(dropna=True),
})

feature_schema

,dtype,missing_rows,missing_pct,unique_values
surface_habitable,int32,0,0.0000,491
n_pieces,float64,3972,0.0962,15
vefa,bool,0,0.0000,2
latitude,float64,0,0.0000,2572476
longitude,float64,0,0.0000,2572476
code_postal,int32,0,0.0000,5636
departement,str,0,0.0000,91


In [ ]:
# PURPOSE:
# Inspect postal-code representation before using it as a geographic
# categorical feature.
#
# Postal codes are geographic labels, not continuous quantities, even
# though the source dataset stores them as integers.

print("Postal code dtype:", model_df["code_postal"].dtype)
print("Unique postal codes:", model_df["code_postal"].nunique())

print("\nExample postal codes:")
print(
    model_df["code_postal"]
    .drop_duplicates()
    .sort_values()
    .head(20)
    .tolist()
)

#### Baseline schema:

Numeric
├── surface_habitable
├── n_pieces
├── latitude
└── longitude

Boolean
└── vefa

Categorical
├── code_postal
└── departement

Routing variable — not predictor
└── type_batiment

Target
└── prix

Dataset management only
└── date_transaction

In [39]:
# PURPOSE:
# Normalize categorical geographic variables before creating the fixed
# modeling datasets.
#
# French postal codes are stored as integers in the source data, which
# removes leading zeros. They are converted to five-character strings.
# Department is also explicitly retained as a categorical string.

model_df["code_postal"] = (
    model_df["code_postal"]
    .astype(str)
    .str.zfill(5)
)

model_df["departement"] = model_df["departement"].astype(str)


# Re-create the specialized populations after the transformation so both
# datasets inherit exactly the same feature representation.
houses_df = (
    model_df.loc[model_df["type_batiment"] == "Maison"]
    .copy()
    .reset_index(drop=True)
)

apartments_df = (
    model_df.loc[model_df["type_batiment"] == "Appartement"]
    .copy()
    .reset_index(drop=True)
)


print("Example postal codes:")
print(
    model_df["code_postal"]
    .drop_duplicates()
    .sort_values()
    .head(20)
    .tolist()
)

print(f"\nHouses:     {len(houses_df):,}")
print(f"Apartments: {len(apartments_df):,}")

Example postal codes:
['01000', '01090', '01100', '01110', '01120', '01130', '01140', '01150', '01160', '01170', '01190', '01200', '01210', '01220', '01230', '01240', '01250', '01260', '01270', '01280']

Houses:     2,329,121
Apartments: 1,801,336


## 7. Split strategy and repeated-property control

The project uses random train, validation, and test partitions because the product is formulated as a cross-sectional valuation problem rather than a time-series forecasting problem.

However, property transactions are not necessarily independent: the same cadastral parcel may appear in the dataset more than once.

If transactions from the same parcel are randomly distributed across training and test sets, model performance could become optimistic because highly similar properties may be represented on both sides of the evaluation.

The cadastral parcel identifier is therefore evaluated as a potential **grouping variable for dataset splitting only**. It will never be used as a predictive feature.

In [40]:
# PURPOSE:
# Measure how frequently cadastral parcels repeat within the selected
# recent metropolitan modeling population.
#
# The parcel identifier is used only to assess split independence.
# It will never become a model predictor.

parcel_check = parquet_file.read(
    columns=[
        "date_transaction",
        "prix",
        "surface_habitable",
        "n_pieces",
        "type_batiment",
        "departement",
        "id_parcelle_cadastre",
    ]
).to_pandas()


# Reproduce the same population-selection rules used for model_df.
parcel_check = parcel_check.loc[
    parcel_check["date_transaction"].between(
        "2020-01-01",
        "2024-06-30"
    )
].copy()

parcel_check = parcel_check.loc[
    ~parcel_check["departement"].isin(overseas_departments)
].copy()

parcel_check = parcel_check.loc[
    parcel_check["prix"] > 0
].copy()

parcel_check = parcel_check.loc[
    parcel_check["surface_habitable"].between(10, 500)
].copy()


# Parcel-frequency diagnostics
parcel_counts = parcel_check["id_parcelle_cadastre"].value_counts(
    dropna=False
)

rows_on_repeated_parcels = parcel_check[
    "id_parcelle_cadastre"
].isin(parcel_counts[parcel_counts > 1].index).sum()


print(f"Modeling rows:             {len(parcel_check):,}")
print(
    f"Unique parcel IDs:         "
    f"{parcel_check['id_parcelle_cadastre'].nunique(dropna=True):,}"
)
print(
    f"Missing parcel IDs:        "
    f"{parcel_check['id_parcelle_cadastre'].isna().sum():,}"
)
print(f"Repeated parcel IDs:       {(parcel_counts > 1).sum():,}")
print(f"Rows on repeated parcels:  {rows_on_repeated_parcels:,}")
print(
    f"Share on repeated parcels: "
    f"{rows_on_repeated_parcels / len(parcel_check) * 100:.2f}%"
)
print(f"Maximum rows per parcel:   {parcel_counts.max():,}")

Modeling rows:             4,148,473
Unique parcel IDs:         2,584,016
Missing parcel IDs:        0
Repeated parcel IDs:       363,623
Rows on repeated parcels:  1,928,080
Share on repeated parcels: 46.48%
Maximum rows per parcel:   535


### Split decision

Repeated cadastral parcels are sufficiently common to affect evaluation design:

- 2,584,016 unique parcel identifiers are observed;
- 363,623 parcel identifiers occur more than once;
- 1,928,080 transactions, representing 46.48% of the modeling population, belong to repeated parcels;
- some parcels contain a large number of transaction records.

A purely row-random split could therefore place transactions associated with the same cadastral parcel in both training and evaluation datasets, producing an overly optimistic estimate of generalization.

The train, validation, and test partitions will consequently use a **random group-aware split based on `id_parcelle_cadastre`**.

The parcel identifier is used exclusively for partitioning. It is not included among the predictive features and will not be required by the production API.

This preserves the cross-sectional random-validation strategy while providing a stronger test of performance on unseen cadastral parcels.

In [41]:
# PURPOSE:
# Rebuild the definitive cleaned modeling population with the cadastral
# parcel ID included exclusively as a split-control variable.
#
# This cell applies Cleaning Policy v2 consistently before creating the
# specialized house and apartment populations.
#
# id_parcelle_cadastre is used ONLY to prevent the same cadastral parcel
# from appearing across train, validation and test sets.
# It will NOT be included in the predictive feature set.
#
# IMPORTANT:
# price_per_m2 is calculated temporarily for transaction-consistency
# cleaning. Because it is derived from the target price, it MUST NOT
# remain in the modeling dataset or become a predictive feature.

modeling_columns_with_group = [
    "date_transaction",
    "prix",
    "surface_habitable",
    "n_pieces",
    "type_batiment",
    "vefa",
    "latitude",
    "longitude",
    "code_postal",
    "departement",
    "id_parcelle_cadastre",
]

model_df = parquet_file.read(
    columns=modeling_columns_with_group
).to_pandas()


# 1. Recent market window
# Keep transactions from January 2020 through June 2024.
model_df = model_df.loc[
    model_df["date_transaction"].between(
        "2020-01-01",
        "2024-06-30"
    )
].copy()


# 2. Metropolitan product scope
# The current product is scoped to metropolitan France.
overseas_departments = {"971", "972", "973", "974"}

model_df = model_df.loc[
    ~model_df["departement"].isin(overseas_departments)
].copy()


# 3. Valid positive target
# Zero or negative transaction prices do not represent the residential
# sale value that the model is intended to estimate.
model_df = model_df.loc[
    model_df["prix"] > 0
].copy()


# 4. Product-scope / plausible habitable surface
# Retain residential properties between 10 and 500 m².
model_df = model_df.loc[
    model_df["surface_habitable"].between(
        10,
        500,
        inclusive="both"
    )
].copy()


# 5. Residential transaction-consistency rule
#
# Diagnostic analysis identified a small number of transactions whose
# recorded price is extremely inconsistent with the habitable surface
# represented by the row.
#
# We retain the deliberately broad range:
#
#     €100/m² <= transaction price per m² <= €30,000/m²
#
# This rule is intended to remove transactions that are incompatible
# with the individual residential-property valuation use case, while
# preserving legitimate geographic variation and expensive/luxury
# residential properties.

model_df["price_per_m2"] = (
    model_df["prix"]
    / model_df["surface_habitable"]
)

before_consistency_filter = len(model_df)

model_df = model_df.loc[
    model_df["price_per_m2"].between(
        100,
        30_000,
        inclusive="both"
    )
].copy()

removed_consistency = (
    before_consistency_filter - len(model_df)
)

removed_consistency_pct = (
    removed_consistency
    / before_consistency_filter
    * 100
)

print(
    "Removed by residential transaction-consistency rule:",
    f"{removed_consistency:,}",
    f"({removed_consistency_pct:.3f}%)"
)


# price_per_m2 is derived from the target and was created ONLY for
# cleaning. Drop it immediately to prevent target leakage.
model_df = model_df.drop(
    columns=["price_per_m2"]
)


# 6. Treat unreliable room counts as missing rather than discarding
# otherwise useful transactions.
model_df["n_pieces"] = model_df["n_pieces"].astype(float)

model_df.loc[
    (model_df["n_pieces"] == 0)
    | (model_df["n_pieces"] > 15),
    "n_pieces"
] = np.nan


# 7. Geographic categorical normalization
#
# Postal codes are represented as five-character strings so that
# leading zeros are preserved.
model_df["code_postal"] = (
    model_df["code_postal"]
    .astype(str)
    .str.zfill(5)
)

model_df["departement"] = (
    model_df["departement"]
    .astype(str)
)


# 8. Create specialized modeling populations.
#
# Property type will later route the prediction to the appropriate
# specialized model rather than being used as a predictor inside
# each model.

houses_df = (
    model_df.loc[
        model_df["type_batiment"] == "Maison"
    ]
    .copy()
    .reset_index(drop=True)
)

apartments_df = (
    model_df.loc[
        model_df["type_batiment"] == "Appartement"
    ]
    .copy()
    .reset_index(drop=True)
)


# 9. Final population checks
print(f"\nMaster population: {len(model_df):,}")
print(f"Houses:            {len(houses_df):,}")
print(f"Apartments:        {len(apartments_df):,}")

print(
    "\nMissing parcel IDs:",
    model_df["id_parcelle_cadastre"].isna().sum()
)

print("\nDate range:")
print(
    model_df["date_transaction"].min(),
    "→",
    model_df["date_transaction"].max()
)

print(
    "\nTemporary price_per_m2 retained:",
    "price_per_m2" in model_df.columns
)

Removed by residential transaction-consistency rule: 18,016 (0.434%)

Master population: 4,130,457
Houses:            2,329,121
Apartments:        1,801,336

Missing parcel IDs: 0

Date range:
2020-01-01 00:00:00 → 2024-06-30 00:00:00

Temporary price_per_m2 retained: False


## 8. Fixed group-aware train / validation / test splits

House and apartment populations are partitioned separately into approximately:

- 70% training data;
- 15% validation data;
- 15% test data.

Splitting is random and reproducible using a fixed random seed, but transactions sharing the same cadastral parcel are kept in the same partition.

This prevents parcel overlap between training and evaluation data while preserving the cross-sectional validation strategy.

The validation set will be used for model comparison and model-selection decisions. The test set remains untouched until the final selected modeling approach is evaluated.

In [42]:
# PURPOSE:
# Create reproducible 70/15/15 train, validation, and test partitions
# while ensuring that the same cadastral parcel never appears in
# more than one partition.
#
# Splits are created separately for houses and apartments.

from sklearn.model_selection import GroupShuffleSplit


def make_group_splits(df, group_col="id_parcelle_cadastre", random_state=42):
    """
    Split a property-type dataset into approximately:
        70% train
        15% validation
        15% test

    Entire cadastral parcels remain within one partition.
    """

    # ---------------------------------------------------------
    # First split:
    # 70% train / 30% temporary evaluation population
    # ---------------------------------------------------------
    train_splitter = GroupShuffleSplit(
        n_splits=1,
        train_size=0.70,
        random_state=random_state
    )

    train_idx, temp_idx = next(
        train_splitter.split(
            df,
            groups=df[group_col]
        )
    )

    train_df = df.iloc[train_idx].copy()
    temp_df = df.iloc[temp_idx].copy()


    # ---------------------------------------------------------
    # Second split:
    # Divide the remaining 30% equally into validation and test.
    #
    # Because this is a group-aware split, 50/50 applies to
    # groups and the final row percentages may differ slightly.
    # ---------------------------------------------------------
    eval_splitter = GroupShuffleSplit(
        n_splits=1,
        train_size=0.50,
        random_state=random_state
    )

    val_idx, test_idx = next(
        eval_splitter.split(
            temp_df,
            groups=temp_df[group_col]
        )
    )

    val_df = temp_df.iloc[val_idx].copy()
    test_df = temp_df.iloc[test_idx].copy()

    return train_df, val_df, test_df

In [43]:
# PURPOSE:
# Apply the exact same splitting methodology independently
# to the house and apartment modeling populations.

house_train, house_val, house_test = make_group_splits(
    houses_df,
    random_state=RANDOM_STATE
)

apartment_train, apartment_val, apartment_test = make_group_splits(
    apartments_df,
    random_state=RANDOM_STATE
)


def print_split_summary(name, full_df, train_df, val_df, test_df):
    """Display row counts and proportions for one property type."""

    total = len(full_df)

    print(f"\n{name}")
    print("-" * 45)

    print(
        f"Train:      {len(train_df):,} "
        f"({len(train_df) / total * 100:.2f}%)"
    )

    print(
        f"Validation: {len(val_df):,} "
        f"({len(val_df) / total * 100:.2f}%)"
    )

    print(
        f"Test:       {len(test_df):,} "
        f"({len(test_df) / total * 100:.2f}%)"
    )

    print(f"Total:      {total:,}")


print_split_summary(
    "HOUSES",
    houses_df,
    house_train,
    house_val,
    house_test
)

print_split_summary(
    "APARTMENTS",
    apartments_df,
    apartment_train,
    apartment_val,
    apartment_test
)


HOUSES
---------------------------------------------
Train:      1,630,944 (70.02%)
Validation: 349,176 (14.99%)
Test:       349,001 (14.98%)
Total:      2,329,121

APARTMENTS
---------------------------------------------
Train:      1,263,001 (70.11%)
Validation: 269,977 (14.99%)
Test:       268,358 (14.90%)
Total:      1,801,336


In [44]:
# PURPOSE:
# Prove that no cadastral parcel leaks across train,
# validation, and test partitions.

def verify_no_group_overlap(
    train_df,
    val_df,
    test_df,
    group_col="id_parcelle_cadastre"
):
    train_groups = set(train_df[group_col])
    val_groups = set(val_df[group_col])
    test_groups = set(test_df[group_col])

    train_val_overlap = len(train_groups & val_groups)
    train_test_overlap = len(train_groups & test_groups)
    val_test_overlap = len(val_groups & test_groups)

    print("Train ↔ Validation overlap:", train_val_overlap)
    print("Train ↔ Test overlap:      ", train_test_overlap)
    print("Validation ↔ Test overlap: ", val_test_overlap)

    assert train_val_overlap == 0
    assert train_test_overlap == 0
    assert val_test_overlap == 0

    print("✓ No cadastral parcel leakage detected.")


print("HOUSES")
verify_no_group_overlap(
    house_train,
    house_val,
    house_test
)

print("\nAPARTMENTS")
verify_no_group_overlap(
    apartment_train,
    apartment_val,
    apartment_test
)

HOUSES
Train ↔ Validation overlap: 0
Train ↔ Test overlap:       0
Validation ↔ Test overlap:  0
✓ No cadastral parcel leakage detected.

APARTMENTS
Train ↔ Validation overlap: 0
Train ↔ Test overlap:       0
Validation ↔ Test overlap:  0
✓ No cadastral parcel leakage detected.


## 9. Persisting the fixed modeling datasets

The group-aware partitions are now frozen and persisted to disk.

Six datasets are stored:

- house train / validation / test;
- apartment train / validation / test.

Each file retains the target and audit variables such as transaction date and cadastral parcel identifier. These audit variables are not predictive features.

Keeping the frozen partitions on disk ensures that all subsequent models are evaluated on exactly the same observations, making model comparisons reproducible and methodologically fair.

The processed datasets are excluded from Git because of their size.

In [45]:
# PURPOSE:
# Save the six frozen modeling partitions as Parquet files.
#
# These files become the single source of truth for all subsequent
# modeling experiments. Every model will use the same observations
# in train, validation, and test.

datasets_to_save = {
    "houses_train.parquet": house_train,
    "houses_validation.parquet": house_val,
    "houses_test.parquet": house_test,
    "apartments_train.parquet": apartment_train,
    "apartments_validation.parquet": apartment_val,
    "apartments_test.parquet": apartment_test,
}

for filename, df in datasets_to_save.items():
    output_path = DATA_PROCESSED_DIR / filename

    df.to_parquet(
        output_path,
        index=False
    )

    print(
        f"{filename:<32} "
        f"{len(df):>10,} rows"
    )

houses_train.parquet              1,630,944 rows
houses_validation.parquet           349,176 rows
houses_test.parquet                 349,001 rows
apartments_train.parquet          1,263,001 rows
apartments_validation.parquet       269,977 rows
apartments_test.parquet             268,358 rows


## 10. Modeling dataset summary

The modeling dataset construction phase produced two specialized residential populations:

- **Houses:** 2,329,121 transactions
- **Apartments:** 1,801,336 transactions
- **Total:** 4,130,457 transactions

The selected market window covers transactions from **January 2020 through June 2024** in metropolitan France.

### Cleaning policy

A conservative cleaning policy was applied consistently before creating the train, validation, and test partitions.

The final population retains:

- transactions with a strictly positive sale price;
- properties with a habitable surface between **10 and 500 m²**;
- transactions between **€100/m² and €30,000/m²**.

The price-per-m² rule was introduced after diagnostic analysis identified a small number of transactions whose recorded price was strongly inconsistent with the individual dwelling represented by the available property characteristics.

This rule removed **18,016 transactions (0.434%)** from the candidate modeling population. The deliberately broad range was chosen to remove clearly inconsistent residential transactions while preserving legitimate geographic variation and high-value properties.

`price_per_m2` is used **only as a data-quality criterion**. Because it is derived from the target (`prix`), it is removed immediately after cleaning and is never used as a model predictor.

Room counts equal to 0 or greater than 15 are treated as missing rather than causing the entire transaction to be discarded.

### Train / validation / test strategy

To reduce evaluation leakage from repeated properties, train, validation, and test datasets are created using **cadastral parcels as grouping units**.

The same cadastral parcel therefore cannot appear in more than one partition.

The final frozen partitions are:

| Property type | Train | Validation | Test | Total |
|---|---:|---:|---:|---:|
| Houses | 1,630,944 (70.02%) | 349,176 (14.99%) | 349,001 (14.98%) | 2,329,121 |
| Apartments | 1,263,001 (70.11%) | 269,977 (14.99%) | 268,358 (14.90%) | 1,801,336 |

Explicit overlap verification confirmed **zero cadastral parcel overlap** between train, validation, and test partitions for both property types.

The cadastral parcel identifier is retained only for audit and split verification and will not be used as a model predictor.

The transaction date is retained for provenance and later robustness diagnostics but is excluded from the initial predictor set.

The six frozen Parquet datasets stored in `data/processed/` are the single source of truth for subsequent model experiments. They will be reused unchanged to ensure fair and reproducible model comparisons.

The **test datasets will remain untouched during model development and model-selection decisions** and will only be used for final evaluation after the modeling approach has been selected.

In [46]:
# PURPOSE:
# Perform a final integrity check on the six persisted modeling datasets.
#
# These Parquet files are the frozen source of truth for all subsequent
# modeling experiments, so we verify:
#   1. every file can be reloaded successfully;
#   2. persisted row counts match the expected frozen partitions;
#   3. the target-derived price_per_m2 variable was NOT persisted;
#   4. cadastral parcel IDs are present for split auditing;
#   5. train/validation/test parcel separation remains intact after saving.

expected_rows = {
    "houses_train.parquet": 1_630_944,
    "houses_validation.parquet": 349_176,
    "houses_test.parquet": 349_001,
    "apartments_train.parquet": 1_263_001,
    "apartments_validation.parquet": 269_977,
    "apartments_test.parquet": 268_358,
}

reloaded_datasets = {}

print("PERSISTED DATASET CHECK")
print("=" * 65)

for filename, expected_count in expected_rows.items():

    file_path = DATA_PROCESSED_DIR / filename
    df_check = pd.read_parquet(file_path)

    reloaded_datasets[filename] = df_check

    # Core integrity assertions
    assert len(df_check) == expected_count
    assert "prix" in df_check.columns
    assert "id_parcelle_cadastre" in df_check.columns
    assert "price_per_m2" not in df_check.columns
    assert df_check["id_parcelle_cadastre"].isna().sum() == 0

    print(
        f"{filename:<32}"
        f"{len(df_check):>10,} rows  ✓"
    )


# ---------------------------------------------------------
# Verify parcel separation again AFTER serialization.
# ---------------------------------------------------------

def persisted_group_overlap(
    train_df,
    val_df,
    test_df,
    group_col="id_parcelle_cadastre"
):
    train_groups = set(train_df[group_col])
    val_groups = set(val_df[group_col])
    test_groups = set(test_df[group_col])

    return {
        "train_validation": len(train_groups & val_groups),
        "train_test": len(train_groups & test_groups),
        "validation_test": len(val_groups & test_groups),
    }


house_overlap = persisted_group_overlap(
    reloaded_datasets["houses_train.parquet"],
    reloaded_datasets["houses_validation.parquet"],
    reloaded_datasets["houses_test.parquet"],
)

apartment_overlap = persisted_group_overlap(
    reloaded_datasets["apartments_train.parquet"],
    reloaded_datasets["apartments_validation.parquet"],
    reloaded_datasets["apartments_test.parquet"],
)


print("\nPARCEL OVERLAP AFTER SERIALIZATION")
print("=" * 65)

print("Houses:")
print(house_overlap)

print("\nApartments:")
print(apartment_overlap)


# Every overlap must remain zero.
assert all(value == 0 for value in house_overlap.values())
assert all(value == 0 for value in apartment_overlap.values())


# Final population consistency.
assert (
    sum(
        len(reloaded_datasets[name])
        for name in [
            "houses_train.parquet",
            "houses_validation.parquet",
            "houses_test.parquet",
        ]
    )
    == 2_329_121
)

assert (
    sum(
        len(reloaded_datasets[name])
        for name in [
            "apartments_train.parquet",
            "apartments_validation.parquet",
            "apartments_test.parquet",
        ]
    )
    == 1_801_336
)


print("\n✓ All six persisted datasets passed the integrity checks.")
print("✓ No target-derived price_per_m2 feature was persisted.")
print("✓ No cadastral parcel leakage detected after serialization.")
print("✓ Cleaning Policy v2 modeling datasets are frozen.")

PERSISTED DATASET CHECK
houses_train.parquet             1,630,944 rows  ✓
houses_validation.parquet          349,176 rows  ✓
houses_test.parquet                349,001 rows  ✓
apartments_train.parquet         1,263,001 rows  ✓
apartments_validation.parquet      269,977 rows  ✓
apartments_test.parquet            268,358 rows  ✓

PARCEL OVERLAP AFTER SERIALIZATION
Houses:
{'train_validation': 0, 'train_test': 0, 'validation_test': 0}

Apartments:
{'train_validation': 0, 'train_test': 0, 'validation_test': 0}

✓ All six persisted datasets passed the integrity checks.
✓ No target-derived price_per_m2 feature was persisted.
✓ No cadastral parcel leakage detected after serialization.
✓ Cleaning Policy v2 modeling datasets are frozen.
